# Usage of Cost Models package

This notebook will go through the currently implemented models to show how they work; This should act as a simple user guide. For more advanced examples like optimization please look at the other notebooks in this directory.

All models have their input defined in the property called `_cm_input_def`, one can inspect the source code and see what inputs are available, their default values and units. Alternatively, you can instantiate a model and call `print_input` method to get an overview of the same information.

Each model constructor can take in inputs in the argument of instantiation. For example, `MyModel(input1=10.0, input2=30.0)` and these are meant to be used as static inputs like energy asset specification.

Moreover, models are meant to be executed with a common `run` method: `MyModel().run(aep=1000)`. In this case the input arguments are dynamic values, that are changing in between model calls. Like in optimization, if one is designing a wind farm layout, the AEP would be different at each iteration depending on the layout. 

### Units

Units are managed with `pint` package; Documentation can be found https://pint.readthedocs.io/en/stable/.

In [1]:
import pint
length = pint.Quantity(1, 'km')
width = pint.Quantity(100, 'm')
length, width

(<Quantity(1, 'kilometer')>, <Quantity(100, 'meter')>)

In [2]:
area = (length * width).to_reduced_units()
area

<Quantity(100000.0, 'meter ** 2')>

Trying to operate on non-matching units will result in dimensionality error. Try uncommenting the cell below and run it:

In [3]:
# length + pint.Quantity(1, "kg")

One can obtain a magnitude of a quantity by accessing an attribute called `m` or `magnitude`; Change the units calling method `to("unit string")` and access the units object with attribute `units`.

In [4]:
area.m, area.magnitude, area.to('km^2').magnitude, area.units

(100000.0, 100000.0, 0.09999999999999999, <Unit('meter ** 2')>)

### PV Cost Model

In [5]:
from costmodels import PVCostModel
from costmodels.units import Quant
import numpy as np

pv_cm = PVCostModel()
pv_cm.print_input()

PVCostModel inputs:
  solar_capacity
	Value: nan
	Type: <class 'pint.registry.Quantity'>
	Unit: MW
  dc_ac_ratio
	Value: 1.5
	Type: <class 'float'>
	Unit: N/A
  panel_cost
	Value: 110000.0
	Type: <class 'pint.registry.Quantity'>
	Unit: EUR/MW
  hardware_installation_cost
	Value: 100000.0
	Type: <class 'pint.registry.Quantity'>
	Unit: EUR/MW
  inverter_cost
	Value: 20000.0
	Type: <class 'pint.registry.Quantity'>
	Unit: EUR/MW
  fixed_onm_cost
	Value: 4500.0
	Type: <class 'pint.registry.Quantity'>
	Unit: EUR/MW


In [6]:
pv_cm = PVCostModel(  # static inputs
    panel_cost=Quant(1e5, "EUR/MW"),
    hardware_installation_cost=Quant(1e5, "EUR/MW"),
    inverter_cost=Quant(1e5, "EUR/MW"),
    fixed_onm_cost=Quant(1e5, "EUR/MW"),
    dc_ac_ratio=1.2,
)

# dynamic inputs
output_capacity_10mw = pv_cm.run(solar_capacity=Quant(10, "MW"))
output_capacity_42mw = pv_cm.run(solar_capacity=Quant(42, "MW"))

print("10 MW PV system cost: ", output_capacity_10mw)
print("42 MW PV system cost: ", output_capacity_42mw)

10 MW PV system cost:  {'capex': <Quantity(3400000.0, 'EUR')>, 'opex': <Quantity(1200000.0, 'EUR')>}
42 MW PV system cost:  {'capex': <Quantity(14280000.0, 'EUR')>, 'opex': <Quantity(5040000.0, 'EUR')>}


### DTU Offshore Cost Model

In [7]:
from costmodels import DTUOffshoreCostModel

# notice no units are specified;
# they are implicitly defined in the class;
# Be careful with calling models in this way
# make sure you are using the correct units!
dtu_offshore_cm = DTUOffshoreCostModel(
    **{
        "rated_power": 5.111111111111111,
        "rotor_diameter": 80,
        "rotor_speed": 9.444444444444445,
        "hub_height": 200.111486515663536,
        "profit": 1.0,
        "capacity_factor": 33,
        "decline_factor": 2.0,
        "nwt": 10,
        "lifetime": 30,
        "wacc": 7.2,
        "inflation": 8,
        "opex": 3.0,
        "devex": 11.11111111111111,
        "abex": 5.555555555555555,
        "water_depth": 33.33333333333333,
        "electrical_cost": 0.0,
        "foundation_option": 1,
        "eprice": 0.2,
        "inflation": 8,
    }
)
cm_output = dtu_offshore_cm.run()
for k in list(cm_output.keys()):
    v = cm_output[k]
    if not isinstance(v, np.ndarray) and not isinstance(v.m, np.ndarray):
        print(k, v)
    else:
        print(k, f"Mean value: {v.mean()}")

production_net 3357780.042130307 MWh
production_discount 3705966.3098985073 MWh
aep_net 111926.00140434357 MWh
aep_discount 123532.21032995025 MWh
devex_net 0.5679012345679012 MEUR
devex_discount 0.6307065679012346 MEUR
capex_discount 83.80852726307309 MEUR
opex_discount 5.134247495653465 MEUR
co2_emission_per_wt Mean value: 1067385.869713646
cost_per_wt Mean value: 3.7505374782137877 MEUR
lcoe 24.170074370989322 EUR/MWh
capex 78.17959632749354 MEUR
opex 17.370092370724095 MEUR
npv 586072893.3903341 EUR
irr 12.559434104834066 %


In [8]:
dtu_offshore_cm.print_input()

DTUOffshoreCostModel inputs:
  rated_power
	Value: 5.111111111111111
	Type: <class 'pint.registry.Quantity'>
	Unit: MW
  rotor_speed
	Value: 9.444444444444445
	Type: <class 'pint.registry.Quantity'>
	Unit: rpm
  rotor_diameter
	Value: 80
	Type: <class 'pint.registry.Quantity'>
	Unit: m
  hub_height
	Value: 200.11148651566353
	Type: <class 'pint.registry.Quantity'>
	Unit: m
  foundation_option
	Value: Foundation.MONOPILE
	Type: <enum 'Foundation'>
	Unit: N/A
  profit
	Value: 1.0
	Type: <class 'pint.registry.Quantity'>
	Unit: %
  capacity_factor
	Value: 33
	Type: <class 'pint.registry.Quantity'>
	Unit: %
  decline_factor
	Value: 2.0
	Type: <class 'pint.registry.Quantity'>
	Unit: %
  nwt
	Value: 10
	Type: <class 'int'>
	Unit: N/A
  wacc
	Value: 7.2
	Type: <class 'pint.registry.Quantity'>
	Unit: %
  devex
	Value: 11.11111111111111
	Type: <class 'pint.registry.Quantity'>
	Unit: EUR/kW
  water_depth
	Value: 33.33333333333333
	Type: <class 'pint.registry.Quantity'>
	Unit: m
  abex
	Value: 5.5

### Minimalistic Cost Model

In [9]:
from costmodels import MinimalisticCostModel

mcm = MinimalisticCostModel()
cmo = mcm.run()
cmo

{'capex': <Quantity(889.213286, 'megaEUR')>,
 'opex': <Quantity(246.720276, 'megaEUR')>,
 'aep': <Quantity(853.444095, 'gigawatt_hour')>,
 'lcoe': <Quantity(372.451, 'EUR / megawatt_hour')>,
 'irr': <Quantity(24.8238563, 'percent')>,
 'npv': <Quantity(2.04324607e+09, 'EUR')>}

### NREL Cost Model

In [10]:
from costmodels import NRELCostModel

nrel_cm = NRELCostModel(
    machine_rating=Quant(5.0, "MW"),
    rotor_diameter=126.0,
    tower_length=90.0,
    blade_number=3,
    blade_has_carbon=False,
    max_tip_speed=80.0,
    max_efficiency=90,
    main_bearing_number=2,
    crane=True,
    eprice=0.2,
    inflation=2,
    nwt=20,
    lifetime=20,
    opex=Quant(20.0, "EUR/kW"),
)
nrel_cm.run(aep=Quant(20.0, "GWh"),)

{'capex': <Quantity(64420707.1, 'EUR')>,
 'opex': <Quantity(100000.0, 'EUR')>,
 'lcoe': <Quantity(75.8475877, 'EUR / megawatt_hour')>,
 'npv': <Quantity(12049881.1, 'EUR')>,
 'irr': <Quantity(3.72555997, 'percent')>}